In [ ]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.append("..")
from src.utils.scene import seed_everything
from itertools import chain
seed_everything(45)

### Generate Layouts for Multi-Object Generation Evaluation Set


In [ ]:
import json
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from src.utils.scene import DiffusionScene
from layouts_utils import gen_layouts_path, gen_renders_path, gen_jsons_path, OBJECTS_CATEGORIES, SCENES, ASPECT_RATIOS, RELATIONS, check_out_of_bounds, check_overlap
   

num_infer_steps = 20
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dtype = torch.float16

# Make directories to save the layouts, renders for debug and jsons
for path in [gen_layouts_path, gen_renders_path, gen_jsons_path]:
    if not path.exists():
        path.mkdir(parents=True)        


num_samples = 100 # Number of layouts to generate
num_seeds = 5 # Number of seeds to generate for each layout

counter = 0
while counter < num_samples:
    layout_dict = {}

    # Build the scene
    scene_size = 5
    print("==> Scene size: ", scene_size)
    scene = DiffusionScene(scene_size=scene_size)
    camera_angle = np.random.randint(70,80)
    print("==> Camera Angle: ", camera_angle)
    scene.move_camera(rotation_angle=camera_angle,rotation_axis=[1,0,0], translation=[0,0,0])
    scene.build_floor(scale_x=2, scale_y=4, floor_offset=-scene_size)
    scene.set_pipe(None, "", num_infer_steps, device, dtype)
    scene_choice = random.choices(SCENES)[0]
    scene_prompt = scene_choice[0]
    scene_categories =  scene_choice[1]
    objects = list(chain(*[OBJECTS_CATEGORIES[c] for c in scene_categories]))

    layout_dict["scene_size"] =  scene_size
    layout_dict["camera_angle"] = camera_angle
    layout_dict["scene"] = scene_prompt
    layout_dict["seeds"] = np.random.randint(0,10000,num_seeds).tolist()
    print("")

    # Define the first box 
    prompt_b1 = random.choice(objects)
    print(f"==> First object is `{prompt_b1}`" )
    aspect_b1 = np.array(ASPECT_RATIOS[prompt_b1])
    print("==> First object aspect ratio is", aspect_b1)
    size_b1 = scene_size * aspect_b1 *2
    print("==> First object size is", size_b1)
    z_b1 = scene_size * np.random.uniform(1.2,2)
    origin_b1 = [0,z_b1,0]
    print("==> First object origin is", origin_b1)


    print("")

    # Define the Second box 
    prompt_b2 = random.choice(objects)
    print(f"==> Second object is `{prompt_b2}`" )
    aspect_b2 = np.array(ASPECT_RATIOS[prompt_b2])
    print("==> Second object aspect ratio is", aspect_b2)
    size_b2 = scene_size * aspect_b2 *2
    print("==> Second object size is", size_b2)
    origin_b2 = [0,z_b1,0]
    print("==> Second object origin is", origin_b2)

    relation = random.choice(RELATIONS[prompt_b2])

    print("relation is :", relation)
    success = False
    for itr in range(20): # Try for 20 times to find a valid layout
        print(itr)
        step = 1 + random.random()
        if relation == "l": # Left
            origin_b1[0] += step
            origin_b2[0] -= step
        elif relation == "r": # Right
            origin_b1[0] -= step
            origin_b2[0] += step
        elif relation == "a":        
            origin_b2[2] = -scene_size + size_b1[2] + size_b2[2] /2 
            print( origin_b2)            

        try:
            scene.add_box(id="box_1", size=size_b1, origin=origin_b1, prompt=prompt_b1)
            mask_b1, latent_mask_b1, p_image_b1 = scene.get_box_masks(box_id="box_1")
            
            scene.add_box(id="box_2", size=size_b2, origin=origin_b2, prompt=prompt_b2)
            mask_b2, latent_mask_b2, p_image_b2 = scene.get_box_masks(box_id="box_2")
        except:
            continue
        
        # Check if the boxes overlap or out of bounds
        if  check_overlap(mask_b1, mask_b2) or check_out_of_bounds(mask_b1) or check_out_of_bounds(mask_b2)  :
            continue
        else:
            success=True
            break

    if not success:
        continue

    rendered_scene = scene.render()
    img_fname = gen_renders_path / f"{counter:04d}.png"
    plt.imsave(img_fname, rendered_scene)

    layout_dict["box_1"] = {"prompt": prompt_b1, "aspect_ratio": aspect_b1.tolist(), "size": size_b1.tolist(), "origin": origin_b1, "relation": None}
    layout_dict["box_2"] = {"prompt": prompt_b2, "aspect_ratio": aspect_b2.tolist(), "size": size_b2.tolist(), "origin": origin_b2, "relation": relation}


    file=open(gen_jsons_path / f"{counter:04d}.json","w")
    json.dump(layout_dict,file,indent=2)
    file.close()


    counter += 1


### Generate Layouts for 3D Translation Consistency Evaluation Set


In [ ]:
import json
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from src.utils.scene import DiffusionScene
from layouts_utils import cons_layouts_path, cons_renders_path, cons_jsons_path, OBJECTS_CATEGORIES, SCENES, ASPECT_RATIOS,  check_out_of_bounds

  

num_infer_steps = 20
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dtype = torch.float16

for path in [cons_layouts_path, cons_renders_path, cons_jsons_path]:
    if not path.exists():
        path.mkdir(parents=True)        


num_samples = 100
counter = 0
while counter < num_samples:
    layout_dict = {}

    # Build the scene
    scene_size = 3
    print("==> Scene size: ", scene_size)
    scene = DiffusionScene(scene_size=scene_size)
    camera_angle = np.random.randint(70,90)
    print("==> Camera Angle: ", camera_angle)
    scene.move_camera(rotation_angle=camera_angle,rotation_axis=[1,0,0], translation=[0,0,0])
    scene.build_floor(scale_x=2, scale_y=2, floor_offset=-scene_size)
    scene.set_pipe(None, "", num_infer_steps, device, dtype)
    
    scene_choice = random.choices(SCENES)[0]
    scene_prompt = scene_choice[0]
    scene_categories =  scene_choice[1]
    objects = list(chain(*[OBJECTS_CATEGORIES[c] for c in scene_categories]))
        
    layout_dict["scene_size"] =  scene_size
    layout_dict["camera_angle"] = camera_angle
    layout_dict["scene"] = scene_prompt
    layout_dict["seeds"] = np.random.randint(0,10000,5).tolist()
    print("")


    # Define the first box 
    prompt_b1 = random.choice(objects)
    print(f"==> First object is `{prompt_b1}`" )
    aspect_b1 = np.array(ASPECT_RATIOS[prompt_b1])
    print("==> First object aspect ratio is", aspect_b1)
    size_b1 = scene_size * aspect_b1 *2
    print("==> First object size is", size_b1)
    z_b1 = scene_size * np.random.uniform(1.2,1.6)
    origin_b1 = [0,z_b1,0]
    print("==> First object origin is", origin_b1)

    try:
        scene.add_box(id="box_1", size=size_b1, origin=origin_b1, prompt=prompt_b1)
        mask_b1, latent_mask_b1, p_image_b1 = scene.get_box_masks(box_id="box_1")
    except:
        continue
    
    ### Add a valid translation    
    selected_action = None
    selected_scale = 0
    for itr in range(10):
        scene.box("box_1").reset()
        action = random.choice(["move_left", "move_right", "zoom_in"])
        action_scale = random.choice([1.0, 1.5, 2.0])

        print(f"==> Performing '{action}' with scale {action_scale}")
        action_fn = getattr(scene.box("box_1"), action)
        action_fn(action_scale)
        mask_b1m, latent_mask_b1m, p_image_b1m = scene.get_box_masks(box_id="box_1")

        if check_out_of_bounds(mask_b1m):
            continue
        else:
            selected_action = action
            selected_scale = action_scale
            break
    
    scene.box("box_1").reset()
    layout_dict["box_1"] = {"prompt": prompt_b1, "aspect_ratio": aspect_b1.tolist(), "size": scene.box("box_1").size.tolist(), "origin": scene.box("box_1").origin, "action": selected_action, "action_scale": selected_scale}

    if selected_action is not None:
        file=open(cons_jsons_path / f"{counter:04d}.json","w")
        json.dump(layout_dict,file,indent=2)
        file.close()

        rendered_scene = scene.render()
        img_fname = cons_renders_path / f"{counter:04d}.png"
        plt.imsave(img_fname, rendered_scene)

    counter += 1